In [1]:
%load_ext autoreload
%autoreload 2
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import normflows as nf
import formula as fo
import summarize as su
import getplot as pl
import time
from torch.distributions import Normal

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class NBase(nn.Module):
    def __init__(self, dim, init_loc=0.0, init_log_scale=-2.5):
        super().__init__()
        self.dim = dim
        self.loc = nn.Parameter(torch.full((dim,), float(init_loc)))
        self.raw_log_scale = nn.Parameter(torch.full((dim,), float(init_log_scale)))

    def log_scale(self):
        return torch.clamp(self.raw_log_scale, min=-5.0, max=2.0)

    def scale(self):
        return torch.exp(self.log_scale())

    def rsample(self, num_samples):
        eta = torch.randn(
            num_samples,
            self.dim,
            device=self.loc.device,
            dtype=self.loc.dtype,
        )
        z0 = self.loc.unsqueeze(0) + self.scale().unsqueeze(0) * eta
        return eta, z0

    def log_prob(self, z0):
        log_scale = self.log_scale().unsqueeze(0)
        var = torch.exp(2.0 * log_scale)
        return -0.5 * (
            ((z0 - self.loc.unsqueeze(0)) ** 2) / var
            + 2.0 * log_scale
            + math.log(2.0 * math.pi)
        ).sum(dim=1)

In [3]:
def simfun1(n=180, p=100, seed=123, snr=3.0, true_prop=0.1, device=None, dtype=torch.float32,):

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    X = rng.standard_normal((n, p)).astype(np.float32)
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)

    n_active = int(p * true_prop)
    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))

    beta_true = np.zeros(p, dtype=np.float32)
    magnitudes = rng.uniform(0.3, 2.0, size=n_active).astype(np.float32)
    signs = rng.choice([-1.0, 1.0], size=n_active).astype(np.float32)
    beta_true[active_idx] = signs * magnitudes

    signal = X @ beta_true
    sigma2 = np.var(signal) / snr
    sigma = np.sqrt(sigma2)

    y = signal + sigma * rng.standard_normal(n).astype(np.float32)
    y = y - y.mean()

    X_t = torch.tensor(X, dtype=dtype, device=device)
    y_t = torch.tensor(y, dtype=dtype, device=device)
    beta_true_t = torch.tensor(beta_true, dtype=dtype, device=device)

    info = {"n": n, "p": p, "n_active": n_active, "sigma2": float(sigma2), "sigma": float(sigma), "active_idx": active_idx, "snr": snr,}

    return X_t, y_t, beta_true_t, info

In [21]:
class AffineCoupling(nn.Module):
    """
    Handwritten masked affine coupling layer,
    but using normflows' MLP as the internal network.
    """
    def __init__(self, dim, mask, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim
        self.register_buffer("mask", mask.view(1, dim))
        self.scale_clip = float(scale_clip)

        widths = [dim] + [hidden_units] * num_hidden_layers + [dim]
        self.s_net = nf.nets.MLP(widths, init_zeros=True)
        self.t_net = nf.nets.MLP(widths, init_zeros=True)

    def forward(self, x, return_logdet=False):
        x_masked = x * self.mask

        raw_log_scale = self.s_net(x_masked) * (1.0 - self.mask)
        shift = self.t_net(x_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        y = x_masked + (1.0 - self.mask) * (x * torch.exp(log_scale) + shift)

        if return_logdet:
            logdet = log_scale.sum(dim=-1)
            return y, logdet
        return y

    def inverse(self, y, return_logdet=False):
        y_masked = y * self.mask

        raw_log_scale = self.s_net(y_masked) * (1.0 - self.mask)
        shift = self.t_net(y_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        x = y_masked + (1.0 - self.mask) * ((y - shift) * torch.exp(-log_scale))

        if return_logdet:
            logdet = (-log_scale).sum(dim=-1)
            return x, logdet
        return x


class FlowMap(nn.Module):
    """
    Same structure as your original TransportMap:
    alternate odd/even masks, stack K affine coupling layers.
    """
    def __init__(self, dim, K=4, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim

        base_mask = torch.tensor(
            [1.0 if i % 2 == 0 else 0.0 for i in range(dim)],
            dtype=torch.float32,
        )

        layers = []
        for k in range(K):
            mask = base_mask if (k % 2 == 0) else (1.0 - base_mask)
            layers.append(
                AffineCoupling(
                    dim=dim,
                    mask=mask,
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, x, return_logdet=False):
        z = x
        if return_logdet:
            total_logdet = x.new_zeros(x.shape[0])

        for layer in self.layers:
            if return_logdet:
                z, logdet = layer(z, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                z = layer(z)

        if return_logdet:
            return z, total_logdet
        return z

    def inverse(self, z, return_logdet=False):
        x = z
        if return_logdet:
            total_logdet = z.new_zeros(z.shape[0])

        for layer in reversed(self.layers):
            if return_logdet:
                x, logdet = layer.inverse(x, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                x = layer.inverse(x)

        if return_logdet:
            return x, total_logdet
        return x

In [22]:
class Relaxedsas(nn.Module):
    def __init__(self, X, y, sigma2, tau, g_theta):
        super().__init__()
        if g_theta is None:
            raise ValueError("g_theta must be provided.")

        self.register_buffer("X", X)
        self.register_buffer("y", y)
        self.register_buffer(
            "sigma2",
            torch.tensor(float(sigma2), dtype=X.dtype, device=X.device),
        )
        self.register_buffer(
            "tau",
            torch.tensor(float(tau), dtype=X.dtype, device=X.device),
        )

        self.n, self.p = X.shape
        self.dim = 2 * self.p + 1
        self.g_theta = g_theta

    def set_tau(self, tau):
        self.tau.fill_(float(tau))

    def decode(self, eps, tau_override=None):
        xi = self.g_theta(eps)

        p = self.p
        s = xi[:, :p]
        u = xi[:, p:2 * p]
        t = xi[:, 2 * p:2 * p + 1]

        tau_use = self.tau if tau_override is None else torch.as_tensor(
            float(tau_override), dtype=eps.dtype, device=eps.device
        )

        gate = torch.sigmoid((u - t) / tau_use)
        beta = s * gate

        return {
            "eps": eps,
            "xi": xi,
            "s": s,
            "u": u,
            "t": t,
            "gate": gate,
            "beta": beta,
        }

    def log_joint(self, eps, tau_override=None):
        dec = self.decode(eps, tau_override=tau_override)
        beta = dec["beta"]

        fit = self.X @ beta.T
        resid = self.y[:, None] - fit

        loglik = -0.5 * (
            resid.pow(2).sum(dim=0) / self.sigma2
            + self.n * torch.log(2.0 * torch.pi * self.sigma2)
        )

        log_p0_eps = -0.5 * (
            eps.pow(2) + math.log(2.0 * math.pi)
        ).sum(dim=1)

        return loglik + log_p0_eps

In [23]:
class FlowVITemp(nn.Module):
    def __init__(self, q0, posterior_flow, generative_model, tau_grid, temp_logits_init=None):
        super().__init__()
        self.q0 = q0
        self.posterior_flow = posterior_flow
        self.generative_model = generative_model

        tau_grid = torch.tensor(tau_grid, dtype=q0.loc.dtype, device=q0.loc.device)
        self.register_buffer("tau_grid", tau_grid)
        self.L = len(tau_grid)

        if temp_logits_init is None:
            temp_logits_init = torch.zeros(self.L, dtype=q0.loc.dtype, device=q0.loc.device)

        self.temp_logits = nn.Parameter(temp_logits_init.clone())

        prior_logits = torch.zeros(self.L, dtype=q0.loc.dtype, device=q0.loc.device)
        self.register_buffer("prior_logits", prior_logits)

    def q_c(self):
        return torch.softmax(self.temp_logits, dim=0)

    def log_p_c(self):
        return torch.log_softmax(self.prior_logits, dim=0)

    def sample_posterior(self, num_samples):
        _, z0 = self.q0.rsample(num_samples)
        eps, logdet = self.posterior_flow(z0, return_logdet=True)
        log_q_eps = self.q0.log_prob(z0) - logdet
        return eps, log_q_eps

    def neg_elbo(self, num_samples=256):
        eps, log_q_eps = self.sample_posterior(num_samples)

        q_c = self.q_c()                 # [L]
        log_p_c = self.log_p_c()         # [L]

        expected_log_joint = 0.0
        for k, tau_k in enumerate(self.tau_grid):
            log_joint_k = self.generative_model.log_joint(eps, tau_override=tau_k)
            expected_log_joint = expected_log_joint + q_c[k] * log_joint_k

        kl_c = torch.sum(q_c * (torch.log(q_c + 1e-12) - log_p_c))

        loss = (log_q_eps - expected_log_joint).mean() + kl_c
        return loss


def build_flow_vi_temp(
    X,
    y,
    sigma2,
    tau_grid,
    K_q=8,
    K_g=8,
    hidden_units=64,
    num_hidden_layers=2,
    scale_clip=2.0,
):
    dim = 2 * X.shape[1] + 1

    g_theta = FlowMap(
        dim=dim,
        K=K_g,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=scale_clip,
    )

    generative_model = Relaxedsas(
        X=X,
        y=y,
        sigma2=sigma2,
        tau=float(tau_grid[-1]),
        g_theta=g_theta,
    )

    q0 = NBase(
        dim=dim,
        init_loc=0.0,
        init_log_scale=-2.5,
    )

    posterior_flow = FlowMap(
        dim=dim,
        K=K_q,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=scale_clip,
    )

    model = FlowVITemp(
        q0=q0,
        posterior_flow=posterior_flow,
        generative_model=generative_model,
        tau_grid=tau_grid,
    )

    return model

In [24]:
def train_flow_temp(
    model,
    epochs=1000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=100,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    grad_hist = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        loss = model.neg_elbo(num_samples=num_samples)
        loss.backward()

        if grad_clip is not None:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            grad_norm = float(grad_norm)
        else:
            grad_norm = 0.0

        optimizer.step()

        losses.append(float(loss.item()))
        grad_hist.append(grad_norm)

        if epoch % print_every == 0 or epoch == 1:
            q_c = model.q_c().detach().cpu().numpy()
            print(
                f"[epoch {epoch:04d}] "
                f"loss={loss.item():.6f} "
                f"grad_norm={grad_norm:.6f} "
                f"q_c={np.round(q_c, 3)}"
            )

    return losses, grad_hist

In [27]:
@torch.no_grad()
def trace_eps_point(model, tau_override=None):
    z0_hat = model.q0.loc.unsqueeze(0)
    eps_hat, _ = model.posterior_flow(z0_hat, return_logdet=True)
    dec = model.generative_model.decode(eps_hat, tau_override=tau_override)

    return {
        "s_hat": dec["s"].detach().cpu(),
        "u_hat": dec["u"].detach().cpu(),
        "t_hat": dec["t"].detach().cpu(),
        "beta_hat": dec["beta"].detach().cpu(),
    }

@torch.no_grad()
def point_selection_summary(point_trace, beta_true):
    u_hat = point_trace["u_hat"].squeeze(0)
    t_hat = point_trace["t_hat"].squeeze(0)
    beta_hat = point_trace["beta_hat"].squeeze(0)

    beta_true_cpu = beta_true.detach().cpu()

    pred = (u_hat > t_hat).int().numpy()
    truth = (beta_true_cpu.abs() > 1e-12).int().numpy()

    tp = int(((pred == 1) & (truth == 1)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    tn = int(((pred == 0) & (truth == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    if t_hat.numel() == 1:
        t_col = np.repeat(t_hat.item(), u_hat.numel())
    else:
        t_col = t_hat.numpy()

    df = pd.DataFrame({"j": np.arange(u_hat.numel()), "beta_true": beta_true_cpu.numpy(),
        "u_hat": u_hat.numpy(), "t_hat": t_col, "beta_hat": beta_hat.numpy(),
        "truth": truth, "pred": pred,
    })

    metrics = {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "n_selected": int(pred.sum()), "selected_idx": np.where(pred == 1)[0].tolist(),
    }

    return df, metrics


@torch.no_grad()
def select_temperature_by_crowding(
    model,
    beta_true=None,
    mass_threshold=0.05,
    top_k=None,
    delta=0.05,
):
    q_c = model.q_c().detach().cpu().numpy()
    tau_grid = model.tau_grid.detach().cpu().numpy()

    candidate_idx = np.where(q_c >= mass_threshold)[0]

    if top_k is not None:
        top_idx = np.argsort(-q_c)[:top_k]
        candidate_idx = np.array(sorted(set(candidate_idx.tolist()) | set(top_idx.tolist())))

    if len(candidate_idx) == 0:
        candidate_idx = np.array([int(np.argmax(q_c))])

    records = []
    for idx in candidate_idx:
        tau_k = float(tau_grid[idx])
        point_trace = trace_eps_point(model, tau_override=tau_k)

        u = point_trace["u_hat"].squeeze(0).numpy()
        t = float(point_trace["t_hat"].squeeze().item())
        marg = u - t

        n_selected = int((marg > 0).sum())
        crowding = int(np.sum(np.abs(marg) < delta))

        records.append({
            "idx": int(idx),
            "tau": tau_k,
            "q_mass": float(q_c[idx]),
            "n_selected": n_selected,
            "crowding": crowding,
            "point_trace": point_trace,
        })

    records = sorted(records, key=lambda d: (d["crowding"], -d["q_mass"]))
    best = records[0]

    if beta_true is not None:
        selection_df, metrics = point_selection_summary(best["point_trace"], beta_true)
        best["selection_df"] = selection_df
        best["metrics"] = metrics

    return best, records

In [29]:
def simflow_temp(
    seed=123,
    device=None,
    dtype=torch.float32,
    n=180,
    p=100,
    snr=3.0,
    true_prop=0.1,
    tau_grid=(1.0, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05),
    K_q=8,
    K_g=8,
    hidden_units=64,
    num_hidden_layers=2,
    epochs=1000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=100,
    delta=0.05,
    mass_threshold=0.05,
    top_k=3,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, y, beta_true, sim_info = simfun1(
        n=n,
        p=p,
        seed=seed,
        snr=snr,
        true_prop=true_prop,
        device=device,
        dtype=dtype,
    )

    print("===== Simulation info =====")
    print(sim_info)

    model = build_flow_vi_temp(
        X=X,
        y=y,
        sigma2=sim_info["sigma2"],
        tau_grid=tau_grid,
        K_q=K_q,
        K_g=K_g,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
    ).to(device)

    losses, grad_hist = train_flow_temp(
        model=model,
        epochs=epochs,
        num_samples=num_samples,
        lr=lr,
        grad_clip=grad_clip,
        print_every=print_every,
    )

    best, records = select_temperature_by_crowding(
        model=model,
        beta_true=beta_true,
        mass_threshold=mass_threshold,
        top_k=top_k,
        delta=delta,
    )

    return {
        "beta_true": beta_true,
        "sim_info": sim_info,
        "model": model,
        "losses": losses,
        "grad_hist": grad_hist,
        "best_temperature": best,
        "temperature_records": records,
    }

In [31]:
out_flow1 = simflow_temp(
    seed=123,
    n=120,
    p=100,
    snr=2.5,
    true_prop=0.1,
    tau_grid=(1.0, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05),
    K_q=8,
    K_g=8,
    hidden_units=64,
    num_hidden_layers=2,
    epochs=6000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=890.626038 grad_norm=197.301971 q_c=[0.143 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 0500] loss=261.323425 grad_norm=99.105103 q_c=[0.142 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 1000] loss=248.476898 grad_norm=39.592304 q_c=[0.142 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 1500] loss=246.986374 grad_norm=39.049629 q_c=[0.142 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 2000] loss=244.360474 grad_norm=40.393723 q_c=[0.142 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 2500] loss=243.467056 grad_norm=55.585476 q_c=[0.143 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 3000] loss=242.516159 grad_norm=35.119865 q_c=[0.143 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 3500] loss=242.369293 grad_norm=38.263683 q_c=[0.143 0.143 0.143 0.143 0.143 0.143 0.143]
[epoch 4000] 

In [20]:
out_flow2 = simflow(
    seed=123,
    n=120,
    p=100,
    snr=2.5,
    true_prop=0.1,
    tau_start=1.0,
    tau_end=0.05,
    warmup_ratio=0.3,
    anneal_ratio=0.7,
    K_q=16,
    K_g=16,
    hidden_units=64,
    num_hidden_layers=2,
    epochs=5000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=891.764465 tau=1.000000 grad_norm=275.459290 n_selected=0 near_boundary=100
[epoch 0500] loss=253.047119 tau=1.000000 grad_norm=82.345413 n_selected=6 near_boundary=1
[epoch 1000] loss=244.918777 tau=1.000000 grad_norm=57.581467 n_selected=6 near_boundary=0
[epoch 1500] loss=243.514114 tau=1.000000 grad_norm=80.642952 n_selected=6 near_boundary=0
[epoch 2000] loss=243.256683 tau=0.652315 grad_norm=71.318977 n_selected=6 near_boundary=0
[epoch 2500] loss=243.093475 tau=0.425151 grad_norm=121.409073 n_selected=7 near_boundary=1
[epoch 3000] loss=243.070496 tau=0.277095 grad_norm=129.240875 n_selected=8 near_boundary=1
[epoch 3500] loss=242.895493 tau=0.180598 grad_norm=195.338684 n_selected=10 near_boundary=3
[epoch 4000] loss=242.897736 tau=0.117706 grad_norm=325